# ApartmentIQ Data

1. Pulling all comspet IDs account has access to
2. Pulling all property IDs associated with that account
3. Sending historical data jobs associated with those property IDs
4. Save files to S3
5. Process read into DBT - following build

Note: Notebook on scheduler is set-up differently

# Pulling compset IDs

In [4]:
import os
import sys
import time
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
TOKEN_URL = "https://data.apartmentiq.io/oauth/token"
BASE_URL = "https://data.apartmentiq.io/apartmentiq/api/v1"
CLIENT_ID = os.getenv("CLIENT_ID")
CLIENT_SECRET = os.getenv("CLIENT_SECRET")

session = requests.Session()
session.headers.update({"User-Agent": "apartmentiq-api-client/1.0"})

token = None
token_expires_at = 0

def get_token():
    global token, token_expires_at
    if token and time.time() < token_expires_at:
        return token
    res = session.post(TOKEN_URL, data={
        "grant_type": "client_credentials",
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
    })
    res.raise_for_status()
    data = res.json()
    token = data["access_token"]
    token_expires_at = time.time() + (data["expires_in"] - 60)
    return token

res = session.get(
    f"{BASE_URL}/accounts",
    headers={"Authorization": f"Bearer {get_token()}"},
)
print(res.json())

{'data': [{'id': 9919636, 'name': 'Affinius Capital'}], 'pagination': {'current_page': 1, 'total_pages': 1, 'total_count': 1, 'per_page': 50}}


In [ ]:
headers = {"Authorization": f"Bearer {get_token()}"}

In [9]:
response = res.json()
affinius_id = response['data'][0]['id']

In [10]:
url_listcompset = f"https://data.apartmentiq.io/apartmentiq/api/v1/accounts/{affinius_id}/comp_sets"
response_listcompset = requests.get(url_listcompset, headers=headers)

In [11]:
data_listset = response_listcompset.json()
df_compsetlist = pd.json_normalize(data_listset['data'])
df_compsetlist.columns = df_compsetlist.columns.str.replace('attributes.', '')

In [14]:
df_compsetlist.head(5)

,id,type,id,name,min_floorplan,max_floorplan,category,created_at,value_add,created_user_id,created_user_name,image,show_recommendations_link,market_survey,custom_property,addresses,subject_property_ids,pricing_approval_stage,owned_property_addresses
0,11861923,comp_set,11861923,101 West End Avenue Market,0,4,market_survey,2024-08-06T19:22:18.000Z,None,104789,Alex Field,https://apartmentiq-assets.rentable.co/images/...,False,True,False,"[101 W End Ave, Manhattan, NY 10023]",[99066266],None,"[{'id': 99066266, 'name': '101 West End Avenue..."
1,11861910,comp_set,11861910,1100 South Market,0,3,market_survey,2024-08-06T18:17:18.000Z,None,104789,Alex Field,https://apartmentiq-assets.rentable.co/images/...,False,True,False,"[1100 South Blvd, Charlotte, NC 28203]",[99002185],None,"[{'id': 99002185, 'name': '1100 South Apartmen..."
2,11861909,comp_set,11861909,19Nineteen Market,0,3,market_survey,2024-08-06T18:15:57.000Z,None,104789,Alex Field,https://apartmentiq-assets.rentable.co/images/...,False,True,False,"[1919 Clarendon Blvd, Arlington, VA 22201]",[99065091],None,"[{'id': 99065091, 'name': '19Nineteen', 'addre..."
3,11888181,comp_set,11888181,216 Pico Boulevard Mixed-Use Complex Report,0,3,research_report,2025-06-03T19:24:14.000Z,False,251219,Brian Griffith,https://apartmentiq-assets.rentable.co/6ece930...,False,False,False,[],[],None,[]
4,11861924,comp_set,11861924,220 East 72 Market,0,4,market_survey,2024-08-06T19:24:08.000Z,None,104789,Alex Field,https://apartmentiq-assets.rentable.co/images/...,False,True,False,"[220 E 72nd St, New York, NY 10021]",[99067024],None,"[{'id': 99067024, 'name': '220 East 72nd Stree..."


In [15]:
compset_ids = df_compsetlist['id'].values[:,1]
print(f'No of Compset IDs available: {len(compset_ids)}')

No of Compset IDs available: 114


# Pulling property IDs

For every compset ID, pulling latest unit-level information to find a list of all unique property IDs we have access to - ultimately we are working with unit level data

In [17]:
arr_propids_raw = []
c = 0
for i in compset_ids:
    c+=1
    print(f'Pulling propertiers for compset ID: S. No {c}, ID: {i}')
    url_listunits = f'''https://data.apartmentiq.io/apartmentiq/api/v1/comp_sets/{i}/units'''
    response_unittest = requests.get(url_listunits, headers=headers)
    data_testunit = response_unittest.json()
    units_df = pd.json_normalize(data_testunit['data'])
    units_df.columns = units_df.columns.str.replace('attributes.', '')
    print(f'No of property IDs found for this compset ID: {len(units_df['property_id'].unique())}')
    arr_propids_raw.append(units_df['property_id'].unique())

Pulling propertiers for compset ID: S. No 1, ID: 11861923
No of property IDs found for this compset ID: 10
Pulling propertiers for compset ID: S. No 2, ID: 11861910
No of property IDs found for this compset ID: 14
Pulling propertiers for compset ID: S. No 3, ID: 11861909
No of property IDs found for this compset ID: 10
Pulling propertiers for compset ID: S. No 4, ID: 11888181
No of property IDs found for this compset ID: 3
Pulling propertiers for compset ID: S. No 5, ID: 11861924
No of property IDs found for this compset ID: 4
Pulling propertiers for compset ID: S. No 6, ID: 11928591
No of property IDs found for this compset ID: 11
Pulling propertiers for compset ID: S. No 7, ID: 11918539
No of property IDs found for this compset ID: 9
Pulling propertiers for compset ID: S. No 8, ID: 11861843
No of property IDs found for this compset ID: 8
Pulling propertiers for compset ID: S. No 9, ID: 11884139
No of property IDs found for this compset ID: 12
Pulling propertiers for compset ID: S. No

In [20]:
arr_propids = []
for x in arr_propids_raw:
    arr_propids +=list(x)

print(f'Final total number of Unique property IDs: {len(set(arr_propids))}')

Final total number of Unique property IDs: 787


In [21]:
# Quick note on above result : there are overlapping property IDs for different compsets

In [22]:
arr_propids = list(set(arr_propids))

# Submitting historical batch jobs for all property IDs to fetch unit level rent/features information

APIQ only allows 200 property IDs per job, earliest date for data pull is 1st January, 2020

In [26]:
for i in range(0,len(arr_propids),200):
    url = "https://data.apartmentiq.io/apartmentiq/api/v1/bulk_api/jobs"

    payload = {
        "report_type": "units", #can also be property/floorplan
        "output_format": "csv",
        "start_date": "2025-06-14",
        "end_date": "2026-06-16",
        "property_ids": [int(x) for x in arr_propids[i:min(i+200,len(arr_propids))]]
    }
    headers = {
        "Authorization": f"Bearer {get_token()}",
        "Content-Type": "application/json"
    }
    
    response = requests.post(url, json=payload, headers=headers)
    
    print(response.text)

{"job_id":"d38d0b64-759a-48e8-984b-a13f5209f949","report_type":"units","status":"submitted","output_format":"csv","property_ids":[99061760,99067905,99098627,99084293,99031046,99069959,99067910,99080199,99139595,99078155,99067918,99018768,99018770,99002390,99080224,99078177,99053602,99072035,99014692,99010596,99076133,99078183,99010592,99010593,99045420,99008560,99078199,99033147,100515902,99010624,99012672,99065920,99072069,99168327,99070024,100513872,99084375,99078236,99217500,99070047,100515936,99068001,100208742,99035242,99002474,100341870,99078256,100524146,99002483,99082357,99078262,99141752,99047547,99031167,99078273,99035267,99037316,99002501,99078278,100255883,99029133,99078286,99068045,99068046,99068049,99068050,99129485,99004568,99012761,99006618,99047579,99008669,99008670,99008677,99004585,99082410,99000493,99000494,99020975,99000496,99000497,100214961,99072179,99008692,99078325,99008693,99033273,99047611,99098811,99002557,99039419,99039423,99043525,99104966,99043527,9903329

## Listing JOB IDs

In [30]:
url_listjobs = "https://data.apartmentiq.io/apartmentiq/api/v1/bulk_api/jobs?page=1&per_page=50"
headers = {"Authorization": f"Bearer {get_token()}"}

response_listjobs = requests.get(url_listjobs, headers=headers)
jobsdata = response_listjobs.json()

In [31]:
job_ids = [job['job_id'] for job in jobsdata['jobs'] if 'job_id' in job]
print(job_ids[:5])

['258a6ce8-89b2-4553-a81e-0fc3fb5531c8', '006dc292-af1f-4cf7-8d8f-0f99054e284a', '4096d6a9-09e7-4b52-9f01-ccfad8a8d7d9', 'd38d0b64-759a-48e8-984b-a13f5209f949', 'e68cba12-142c-459c-b6ab-630da39ef3c3']


## Listing Job Status

In [38]:
import requests

url = "https://data.apartmentiq.io/apartmentiq/api/v1/bulk_api/jobs/{job_id}"

for x in job_ids[:4]:
    url = f"https://data.apartmentiq.io/apartmentiq/api/v1/bulk_api/jobs/{x}"
    response = requests.get(url, headers=headers)
    response_print = response.json()
    print(f'Job ID: {response_print['job_id']}')
    print(f'Status: {response_print['status']}')
    print(f'Error message: {response_print['error_message']}')


Job ID: 258a6ce8-89b2-4553-a81e-0fc3fb5531c8
Status: succeeded
Error message: None
Job ID: 006dc292-af1f-4cf7-8d8f-0f99054e284a
Status: succeeded
Error message: None
Job ID: 4096d6a9-09e7-4b52-9f01-ccfad8a8d7d9
Status: succeeded
Error message: None
Job ID: d38d0b64-759a-48e8-984b-a13f5209f949
Status: succeeded
Error message: None


# Saving data extracted to S3

only run below, once job status for all jobs submitted is succeeded

If a particularly large job has been submitted - best to download job results https://developers.apartmentiq.io/api-reference/bulk-data-export/download-batch-job-results

In [39]:
import io

final_df = pd.DataFrame()
for i in range(4):
    url_results_job = "https://data.apartmentiq.io/apartmentiq/api/v1/bulk_api/jobs/"+ job_ids[i] + "/results"
    headers = {"Authorization": f"Bearer {get_token()}"}
    response_prop = requests.get(url_results_job, headers=headers)
    df_prop= pd.read_csv(io.StringIO(response_prop.text), on_bad_lines='skip')
    print(df_prop)
    final_df = pd.concat([final_df,df_prop],axis=0)

        report_generation_date  market_id  \
0                   2025-06-14      93676   
1                   2025-06-14      93676   
2                   2025-06-14      93676   
3                   2025-06-14      93676   
4                   2025-06-14      94229   
...                        ...        ...   
1912501             2026-06-16      94266   
1912502             2026-06-16      93676   
1912503             2026-06-16      93676   
1912504             2026-06-16      93741   
1912505             2026-06-16      93561   

                                             market_name  property_id  \
0          Chicago-Naperville-Elgin, IL-IN-WI Metro Area     99031039   
1          Chicago-Naperville-Elgin, IL-IN-WI Metro Area     99031039   
2          Chicago-Naperville-Elgin, IL-IN-WI Metro Area     99031039   
3          Chicago-Naperville-Elgin, IL-IN-WI Metro Area     99031039   
4        Riverside-San Bernardino-Ontario, CA Metro Area     99006163   
...                  

In [48]:
int_cols_with_nas = [
    "year_built",
    "year_renovated",
    "sq_ft",
    "rent_change_count",
    "row_index",
]
for col in int_cols_with_nas:
    if col in final_df.columns:
        final_df[col] = final_df[col].astype("Int64")

if "zip_code" in final_df.columns:
    final_df["zip_code"] = (
        final_df["zip_code"]
        .fillna("")
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
    )

if "concession_details" in final_df.columns:
    final_df["concession_details"] = final_df["concession_details"].fillna("")

print("Pandas schema cleanup complete!")


Pandas schema cleanup complete!


In [ ]:
import boto3
s3_client = boto3.client('s3')
bucket_name = 'apartmentiq-region1'
s3_file_key = 'units-raw-history/2025units.csv'

# 4. Save DataFrame to an in-memory buffer, then upload directly to S3
csv_buffer = io.StringIO()
final_df.to_csv(csv_buffer, index=False)

s3_client.put_object(
    Bucket=bucket_name, 
    Key=s3_file_key, 
    Body=csv_buffer.getvalue()
)

print(f"Successfully saved file to s3://{bucket_name}/{s3_file_key}")


In [51]:
final_df[final_df['unit_code'] == '99083916_MFTEIncomeRestrictedSEDUModernStudioApartmentwithInUnitWasherDryerGarageParkingPetFriendlyLivingTournow']


,report_generation_date,market_id,market_name,property_id,property_name,property_url,address,city,state,zip_code,...,concessions,concession_details,last_change_date,last_change_amount,total_change_amount,rent_change_count,trucomp_flag,trucomp_rent_diff_pct,trucomp_sq_ft_diff_pct,trucomp_rent_psf_diff_pct
1826045,2026-06-02,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
1830898,2026-06-03,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
1840158,2026-06-04,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
1842810,2026-06-05,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
1849087,2026-06-06,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
1856565,2026-06-07,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
1861694,2026-06-08,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
1867856,2026-06-09,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
1874692,2026-06-10,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
1881375,2026-06-11,94288,"Seattle-Tacoma-Bellevue, WA Metro Area",99083916,Flow Eastlake Apartments,https://www.floweastlake.com/,3150 Fairview Ave E,Seattle,WA,98102,...,NaN,,NaN,NaN,NaN,0,False,NaN,NaN,NaN
